# Independent fusion search for flip-averaged embeddings

Run after flip extraction and combination. The previous original-feature winner is evaluated first on the new representation; **94.75% is a historical comparison, not an expected new score**. No official test features or labels are loaded. The prior completed search ledger and `fusion_search.py` are read only; the new ledger lives inside this flip experiment directory.

The 25 initial candidates include the previous winner, followed by one frozen local refinement list of 10–20 candidates. A two-hour cumulative SVM budget and incremental, resumable checkpoints apply. A measured score above 95% pauses for review; at least 95.5% stops automatically. The assignment still requires **strictly >95% official test accuracy**, which this notebook cannot establish.

All expensive execution switches default to `False`. Keep the same experiment ID as the extraction and combination notebooks.


In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "flip_features.py").is_file():
    ROOT = ROOT / "tugas-7"
if not (ROOT / "flip_features.py").is_file():
    raise FileNotFoundError("Run from the repository root or tugas-7")
ROOT = ROOT.resolve()
sys.path.insert(0, str(ROOT))
EXPERIMENT_ID = "fusion_flip_v1_001"
import pandas as pd
from fusion_search import prepare_features, svm_evaluator, ranked
from flip_fusion_search import (
    FlipFusionExperiment, HISTORICAL_CORRECT, PREVIOUS_WINNER,
    load_flip_development, new_search_directory, read_historical_result,
)

RUN_PREVIOUS_WINNER = True
RUN_SEARCH = True
CONTINUE_AFTER_REVIEW = True

REVIEW_NOTE = (
    "The baseline reached 95.03% validation accuracy. "
    "Continue the existing bounded SVM search to investigate "
    "whether it can improve toward 95.5%. "
    "Additional validation tuning carries overfitting risk. "
    "Do not expand the search budget or use official test data."
)

print("New ledger:", new_search_directory(ROOT, EXPERIMENT_ID) / "search_state.json")
print("Previous winner:", PREVIOUS_WINNER)


New ledger: /home/saga/anaconda3/envs/envGPU/BDA_Project/tugas-7/feature-engineering/experiments/fusion_flip_v1_001/svm_search/search_state.json
Previous winner: {'weights': [1.0, 0.5, 1.0], 'C': 0.5477225575051661, 'gamma': 0.00025}


## 1. Measure the previous winner on averaged features

This first fit establishes the new representation's baseline. It uses 40K training rows for `VarianceThreshold`, `StandardScaler`, and SVM fitting. Validation rows are transformed only; block weights are applied after scaling. The exact result is saved to the new ledger before search starts.


In [2]:
experiment = None
evaluate = None
if RUN_PREVIOUS_WINNER:
    data, provenance, historical = load_flip_development(ROOT, EXPERIMENT_ID)
    X_train, X_val, block_ids = prepare_features(data)
    evaluate = svm_evaluator(X_train, X_val, data["y_train"], data["y_val"], block_ids)
    experiment = FlipFusionExperiment(
        new_search_directory(ROOT, EXPERIMENT_ID), provenance, historical,
    )
    display(experiment.reproduce_baseline(evaluate))
else:
    print("Previous winner has not been evaluated on flip embeddings by this notebook.")


{'stop_reason': 'previous_winner_measured_on_flip',
 'completed_candidates': 1,
 'compute_seconds': 7.800527038999462,
 'remaining_seconds': 7192.1994729610005,
 'best': {'attempts': [{'duration_seconds': 7.800527038999462,
    'started_at': '2026-10-08T01:51:05.940154+00:00',
    'status': 'completed'}],
  'candidate_id': '3042501f1a7f35f6111f',
  'correct': 9503,
  'duration_seconds': 7.800527038999462,
  'effective_gamma': 0.00025,
  'fit_seconds': 5.1894495239994285,
  'parameters': {'C': 0.5477225575051661,
   'gamma': 0.00025,
   'weights': [1.0, 0.5, 1.0]},
  'prediction_seconds': 2.429414359000475,
  'status': 'completed',
  'support_vectors': 469,
  'validation_accuracy': 0.9503,
  'validation_rows': 10000},
 'official_test': 'Not evaluated. Prior individual-model test results were already observed.',
 'scope': 'Development selection only; no final 50K fit or test-ready model is saved.',
 'historical_original': {'ledger_path': '/home/saga/anaconda3/envs/envGPU/BDA_Project/tuga

## 2. Budgeted joint fusion search

Enable only after inspecting the first measured result. The search skips completed candidates and refuses a changed input, helper, environment, or policy fingerprint. It stops after the planned initial/refinement lists or when its cumulative budget is spent. If it pauses above 95%, decide whether to stop or justify continuing the remaining bounded list; do not repeatedly tune against validation without reviewing generalization risk.


In [3]:
if RUN_SEARCH:
    if experiment is None:
        raise RuntimeError("Run the previous-winner cell first")
    display(experiment.run_search(evaluate))
else:
    print("SVM optimization disabled.")


{'stop_reason': 'review_above_95_before_more_search',
 'completed_candidates': 1,
 'compute_seconds': 7.800527038999462,
 'remaining_seconds': 7192.1994729610005,
 'best': {'attempts': [{'duration_seconds': 7.800527038999462,
    'started_at': '2026-10-08T01:51:05.940154+00:00',
    'status': 'completed'}],
  'candidate_id': '3042501f1a7f35f6111f',
  'correct': 9503,
  'duration_seconds': 7.800527038999462,
  'effective_gamma': 0.00025,
  'fit_seconds': 5.1894495239994285,
  'parameters': {'C': 0.5477225575051661,
   'gamma': 0.00025,
   'weights': [1.0, 0.5, 1.0]},
  'prediction_seconds': 2.429414359000475,
  'status': 'completed',
  'support_vectors': 469,
  'validation_accuracy': 0.9503,
  'validation_rows': 10000},
 'official_test': 'Not evaluated. Prior individual-model test results were already observed.',
 'scope': 'Development selection only; no final 50K fit or test-ready model is saved.',
 'historical_original': {'ledger_path': '/home/saga/anaconda3/envs/envGPU/BDA_Project/tu

## 3. Optional continuation after a >95% review

Leave disabled unless the search paused strictly above 95% and below 95.5%. A written review records the decision. This does not reset the two-hour budget or add search rounds.


In [4]:
if CONTINUE_AFTER_REVIEW:
    if experiment is None or not RUN_SEARCH:
        raise RuntimeError("Run the previous-winner and search cells first")
    display(experiment.run_search(evaluate, review_note=REVIEW_NOTE))


6a760c66307d7c5b0b26: 95.0000%, 19.3s, {'C': 0.1, 'gamma': 0.0005, 'weights': [1.0, 0.5, 1.0]}
1102639473d298c57098: 95.0100%, 8.5s, {'C': 0.3, 'gamma': 0.0005, 'weights': [1.0, 0.5, 1.0]}
bc241ea4dcf5e9c13cee: 95.0500%, 8.3s, {'C': 1.0, 'gamma': 0.0001, 'weights': [1.0, 0.5, 1.0]}
8c646eebae5a16b2f264: 95.0000%, 5.4s, {'C': 1.0, 'gamma': 0.00025, 'weights': [1.0, 0.5, 1.0]}
d8c365ddd936b1bcc4c0: 95.0500%, 11.3s, {'C': 0.3, 'gamma': 0.00025, 'weights': [1.0, 0.5, 1.0]}
d981c5ee8c1c10c6c14e: 95.0300%, 7.4s, {'C': 10.0, 'gamma': 'scale', 'weights': [1.0, 0.75, 0.75]}
aa1fee5336f0dea3ebd2: 95.0000%, 10.7s, {'C': 1.0, 'gamma': 0.0001, 'weights': [1.0, 0.75, 0.25]}
1f349fcfb63efba4d870: 95.0100%, 7.0s, {'C': 1.0, 'gamma': 'scale', 'weights': [1.0, 0.75, 0.5]}
077c70d7f5f8c708f29f: 94.9800%, 6.6s, {'C': 1.0, 'gamma': 0.00025, 'weights': [1.0, 0.5, 0.25]}
b4a29cb9dc2215c24895: 94.7500%, 15.3s, {'C': 0.1, 'gamma': 0.001, 'weights': [1.0, 1.0, 0.5]}
9eebaa0977555738ee3b: 94.9600%, 4.2s, {'C': 3

{'stop_reason': 'bounded_search_complete_review_results',
 'completed_candidates': 45,
 'compute_seconds': 445.75421114600067,
 'remaining_seconds': 6754.245788853999,
 'best': {'candidate_id': 'c535509d716ce6e867bf',
  'parameters': {'C': 1.0,
   'gamma': 0.00015811388300841897,
   'weights': [1.0, 0.5, 0.5]},
  'attempts': [{'status': 'completed',
    'started_at': '2026-10-08T02:02:49.277318+00:00',
    'duration_seconds': 6.9449183819997415}],
  'status': 'completed',
  'correct': 9515,
  'validation_rows': 10000,
  'fit_seconds': 4.4551013350001085,
  'prediction_seconds': 2.3280713970007128,
  'effective_gamma': 0.00015811388300841897,
  'support_vectors': 515,
  'validation_accuracy': 0.9515,
  'duration_seconds': 6.9449183819997415},
 'official_test': 'Not evaluated. Prior individual-model test results were already observed.',
 'scope': 'Development selection only; no final 50K fit or test-ready model is saved.',
 'historical_original': {'ledger_path': '/home/saga/anaconda3/env

## 4. Compare with the saved original-feature result

Historical data comes from the completed original ledger. Flip data stays **Pending** until this notebook actually measures it. On a validation tie, prefer original embeddings because one CNN view per image is cheaper. This comparison is not official test performance.


In [5]:
historical = read_historical_result(ROOT)
summary = experiment.summary() if experiment is not None else None
best = summary["best"] if summary else None
comparison = pd.DataFrame({
    "Metric": ["Validation accuracy", "Correct predictions", "Best block weights",
               "C", "gamma", "Extraction method", "Extraction duration (s)",
               "SVM search duration (s)"],
    "Original embeddings — saved history": [
        f"{historical['accuracy']:.4%}", f"{historical['correct']}/10000",
        tuple(historical["parameters"]["weights"]), historical["parameters"]["C"],
        historical["parameters"]["gamma"], historical["extraction_method"],
        "Historical/not recorded", historical["search_seconds"],
    ],
    "Flip-averaged embeddings — this experiment": [
        f"{best['validation_accuracy']:.4%}" if best else "Pending",
        f"{best['correct']}/10000" if best else "Pending",
        tuple(best["parameters"]["weights"]) if best else "Pending",
        best["parameters"]["C"] if best else "Pending",
        best["parameters"]["gamma"] if best else "Pending",
        "Original + horizontal flip", summary["flip_extraction_seconds"] if best else "Pending",
        summary["compute_seconds"] if best else "Pending",
    ],
})
display(comparison)
if best:
    print("Difference vs 9475 original correct:", best["correct"] - HISTORICAL_CORRECT)
    print("Validation preference (ties favor original):", summary["preferred_representation"])
    print("Stop reason:", summary["stop_reason"])
else:
    print("Flip result pending; no new accuracy has been measured here.")


,Metric,Original embeddings — saved history,Flip-averaged embeddings — this experiment
0,Validation accuracy,94.7500%,95.1500%
1,Correct predictions,9475/10000,9515/10000
2,Best block weights,"(1.0, 0.5, 1.0)","(1.0, 0.5, 0.5)"
3,C,0.547723,1.0
4,gamma,0.00025,0.000158
5,Extraction method,original v3 embeddings,Original + horizontal flip
6,Extraction duration (s),Historical/not recorded,112.083438
7,SVM search duration (s),496.382332,445.754211


Difference vs 9475 original correct: 40
Validation preference (ties favor original): flip-averaged
Stop reason: bounded_search_complete_review_results


## Stop for review

This stage ends with the development comparison. It does not retrain CNNs, perform a final 50K SVM fit, or run official testing. Review the measured comparison before considering another experiment.
